<a href="https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/NB03_general.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. First-time setup — create the Drive folders

Run this **once** to build the folder structure on your Drive, then drop your inputs in. Safe to
re-run. The two location lines below are the **only** place `PROJECT_DRIVE` and `COLLECTION` are set —
every path in the notebook derives from them, so change them here and nowhere else.

In [ ]:
Cfrom google.colab import drive
drive.mount("/content/drive")
from pathlib import Path

# ---- LOCATION (the ONLY place these are set) ----
PROJECT_DRIVE = "/content/drive/MyDrive/BIOINFO4-5203-F26"   # your project folder on Drive
COLLECTION    = "NB03_taxon_directed_homologs"              # supervisory folder (your existing Data/Outputs folder)

PROJECT = Path(PROJECT_DRIVE)
_folders = {
    "Data / inputs":            PROJECT / "Data" / COLLECTION,
    "Databases / shared cache": PROJECT / "Databases" / "proteome_library",
    "   proteomes":             PROJECT / "Databases" / "proteome_library" / "proteomes",
    "   blastdbs":              PROJECT / "Databases" / "proteome_library" / "blastdbs",
    "   manifest":              PROJECT / "Databases" / "proteome_library" / "manifest",
    "Outputs / results":        PROJECT / "Outputs" / COLLECTION,
    "Notebooks":                PROJECT / "Notebooks",
    "Reports":                  PROJECT / "Reports",
}
for _p in _folders.values():
    _p.mkdir(parents=True, exist_ok=True)

# drop-in reminders. The taxon table is SHARED, so it lives with the library, not in a project folder.
_db = PROJECT / "Databases" / "proteome_library"
(_db / "PUT_TAXON_TABLE_HERE.txt").write_text(
    "Drop the taxon table CSV in THIS folder (the shared library), e.g. ComplexI_library_flat.csv,\n"
    "then set LIBRARY_CSV in Section 1 to its filename. Every protein search shares this one table.\n",
    encoding="utf-8")
(PROJECT / "Data" / COLLECTION / "README.txt").write_text(
    "Each protein you search gets its OWN subfolder here, named by QUERY_NAME (set in Section 1).\n"
    "Section 2 creates Data/" + COLLECTION + "/<QUERY_NAME>/ ; drop that protein's query FASTA there.\n",
    encoding="utf-8")

print("Folders ready under:", PROJECT)
for _label, _p in _folders.items():
    print(f"  {_label:26s} {_p}")
print("\nDrop the shared taxon table here (once):")
print(f"  - taxon table  ->  {_db}")
print("Per-protein query folders are created by Section 2 once you set QUERY_NAME.")
print("Then edit Section 1 and run it onward.")


## 1. Configuration and one-time setup

Builds a **local proteome library** (one BLAST database per taxon, cached on Drive) and
searches it with **one protein query at a time**. No remote BLAST queue, so no throttling.
The engine is protein-agnostic — the same library serves Complex I subunits, cytochromes, or
anything else; only the query changes.

The library (proteomes + databases) is built once and shared; each protein is a separate *run*
that reuses it. To search another protein, change `QUERY_FASTA` + `QUERY_NAME` and re-run from
section 6 — the library is not rebuilt. To grow the database, add rows to the taxon table (or
drop a `<taxid>.faa` into `Databases/proteome_library/proteomes/`) and re-run section 4.

**Drive layout** — a generic supervisory folder (`COLLECTION`) with a per-protein subfolder
(`QUERY_NAME`) under both Data and Outputs, and one shared library:
- `Data/<COLLECTION>/<QUERY_NAME>/` — inputs you drop in: this protein's query FASTA
- `Databases/proteome_library/` — the shared cache: the taxon table, `proteomes/`, `blastdbs/`, `manifest/` (built once, read by every protein)
- `Outputs/<COLLECTION>/<QUERY_NAME>/` — this protein's results: summary CSV, hits FASTA, logs, checkpoints
- code lives in your `Notebooks/` folder; commit this notebook to your repo by hand

Edit the required settings, then run every cell in order once. Re-running any cell is safe:
completed work is cached and skipped.

In [ ]:
import subprocess, sys, os

# --- one-time dependency setup (idempotent) ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython==1.85"], check=True)
if not os.path.exists("/usr/local/bin/datasets"):
    subprocess.run("curl -sSL 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets'"
                   " -o /usr/local/bin/datasets && chmod +x /usr/local/bin/datasets", shell=True, check=True)
if subprocess.run("which blastp", shell=True, capture_output=True).returncode != 0:
    subprocess.run("apt-get -qq update && apt-get -qq install -y ncbi-blast+", shell=True, check=True)
from Bio import Entrez

# ================= REQUIRED SETTINGS =================
# PROJECT_DRIVE and COLLECTION are set in Section 0 (the only place). These are the per-run choices:
LIBRARY_CSV   = "Taxon_library_v1.csv"   # taxon table; SHARED, in Databases/proteome_library/
QUERY_FASTA   = "Cyt-c_cyano.faa"               # this protein's query; in Data/<COLLECTION>/<QUERY_NAME>/
QUERY_NAME    = "CytC"                          # the protein you are searching now; names the sub-folders
Entrez.email  = "rob.burnap@okstate.edu"           # required by NCBI

# ================= OPTIONAL SETTINGS =================
NCBI_API_KEY = ""                       # recommended; speeds up name/taxid lookups
Entrez.api_key = NCBI_API_KEY or None
Entrez.tool = COLLECTION

TOP_N = 3                               # keep top-N hits per taxon (paralogs), not just the best
EVALUE = 1e-2                           # hit threshold
MIN_QUERY_COVERAGE = 0.0                # percent of the query covered by the best HSP (0 disables)

TAXID_COLUMN = "auto"                   # auto -> resolved_taxid, else species_taxid, else NCBI_TaxID
ASSEMBLY_SOURCE = "RefSeq"              # RefSeq (annotated) or GenBank
ALLOW_NONREF_FETCH = False             # if no reference/representative genome, do NOT grab a random assembly

SEARCH_ENGINE = "blastp"               # blastp (BLAST+); DIAMOND can be added later
FORCE_REFETCH = False                  # re-download proteomes even if cached
FORCE_REBUILD_DB = False               # rebuild BLAST dbs even if present
MAX_RETRIES = 3
BACKOFF_START = 5.0


## 2. Validate settings, mount Drive, prepare the workspace

Mounts Drive and creates the per-protein input/output folders (`Data/<COLLECTION>/<QUERY_NAME>/`
and `Outputs/<COLLECTION>/<QUERY_NAME>/`). The `proteomes/` and `blastdbs/` databases and the
taxon table are **shared** (read from `Databases/proteome_library/`). Loads the taxon table and
picks the TaxID column (preferring a verified one if present).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import pandas as pd, json

PROJECT = Path(PROJECT_DRIVE)
# Per-protein inputs/outputs nest under COLLECTION/QUERY_NAME; the library + taxon table are shared:
DATA_ROOT = PROJECT / "Data"      / COLLECTION / QUERY_NAME   # THIS protein's query FASTA
OUT_ROOT  = PROJECT / "Outputs"   / COLLECTION / QUERY_NAME   # THIS protein's results
DB_ROOT   = PROJECT / "Databases" / "proteome_library"       # SHARED: taxon table + proteomes + dbs

INPUT_DIR    = DATA_ROOT
PROTEOME_DIR = DB_ROOT / "proteomes"   # shared: <taxid>.faa
DB_DIR       = DB_ROOT / "blastdbs"    # shared: <taxid>.p*
RUNS_DIR     = OUT_ROOT
STATE_DIR    = RUNS_DIR / "state"      # per-taxon search checkpoints for THIS protein
for d in (DATA_ROOT, PROTEOME_DIR, DB_DIR, DB_ROOT / "manifest", RUNS_DIR, STATE_DIR):
    d.mkdir(parents=True, exist_ok=True)

EVENTS         = RUNS_DIR / "events.jsonl"
SUMMARY_CSV    = RUNS_DIR / f"{QUERY_NAME}_summary.csv"
HITS_FASTA     = RUNS_DIR / f"{QUERY_NAME}_hits.faa"
LIB_STATUS_CSV = DB_ROOT / "manifest" / "library_status.csv"  # lives with the shared library

if "@" not in Entrez.email or Entrez.email == "your.name@your.institution.edu":
    raise ValueError("Set Entrez.email in the configuration cell before continuing.")

def _resolve_input(name):
    p = Path(name)
    return p if p.is_absolute() else DATA_ROOT / name   # this protein's query lives here

lib_path = DB_ROOT / LIBRARY_CSV                          # taxon table is SHARED, with the library
if not lib_path.exists():
    raise FileNotFoundError(f"Taxon table not found at {lib_path}. Drop it into the shared library "
                            f"folder {DB_ROOT} (created by Section 0).")
library = pd.read_csv(lib_path)

def _pick_taxid_col(df):
    if TAXID_COLUMN != "auto":
        if TAXID_COLUMN not in df.columns:
            raise ValueError(f"TAXID_COLUMN '{TAXID_COLUMN}' not in CSV columns {list(df.columns)}")
        return TAXID_COLUMN
    for c in ("resolved_taxid", "species_taxid", "NCBI_TaxID", "taxid"):
        if c in df.columns:
            return c
    raise ValueError(f"No taxid column found in {list(df.columns)}")
TAXCOL = _pick_taxid_col(library)

def _as_taxid(v):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return None

targets, skipped = [], []
for _, r in library.iterrows():
    tx = _as_taxid(r.get(TAXCOL))
    label = str(r.get("Organism", r.get("organism", f"taxid_{tx}"))).strip()
    if tx is None:
        skipped.append(label); continue
    targets.append({
        "taxid": tx,
        "label": label,
        "tier": r.get("Tier", ""),
        "species_taxid": _as_taxid(r.get("species_taxid")),
        "proteome_file": (str(r["proteome_file"]).strip()
                          if "proteome_file" in library.columns and pd.notna(r.get("proteome_file")) else ""),
    })

print(f"Library: {len(targets)} taxa with a TaxID (column '{TAXCOL}').")
if skipped:
    print(f"  {len(skipped)} row(s) skipped for missing TaxID - fix these in the verify step first:")
    for s in skipped[:20]:
        print("   -", s)


## 3. Library + search helpers

Fetching a proteome, building its database, and searching it are separate functions so each
can be swapped independently. `ensure_proteome` honours an existing `<taxid>.faa` (whether you
dropped one in by hand or it was fetched earlier) and a `proteome_file` override column, so you
can substitute a curated proteome without touching the code. Full hit sequences are pulled from
the **local** proteome, so there is no per-hit network call.

In [ ]:
import time, json, shutil, tempfile, zipfile, glob, hashlib, random
from datetime import datetime, timezone
from Bio import SeqIO

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def atomic_text(path, text):
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)

def save_json(path, data):
    atomic_text(path, json.dumps(data, indent=2, ensure_ascii=False) + "\n")

def event(kind, **details):
    with EVENTS.open("a", encoding="utf-8") as h:
        h.write(json.dumps({"time": utc_now(), "kind": kind, **details}) + "\n")

def retry_call(fn, *, what):
    for attempt in range(MAX_RETRIES + 1):
        try:
            return fn()
        except Exception as exc:
            if attempt == MAX_RETRIES:
                raise
            delay = min(120, BACKOFF_START * 2 ** attempt) + random.uniform(0, 1)
            event("retry", operation=what, attempt=attempt + 1, delay=delay, error=str(exc))
            print(f"  {what}: retry {attempt + 1}/{MAX_RETRIES} in {delay:.0f}s")
            time.sleep(delay)

def run_cmd(cmd, timeout=1800):
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    return p.returncode, p.stdout, p.stderr

# ---- proteomes ----
def proteome_path(taxid):
    return PROTEOME_DIR / f"{taxid}.faa"

def _download_proteome(taxid, species_taxid=None):
    """Fetch one annotated proteome via NCBI datasets, most-specific first:
    strain reference -> species reference -> strain latest annotated (RefSeq) ->
    strain latest annotated (GenBank). Most bacterial strains have no 'reference'
    designation, so the annotated-latest fallbacks are what actually succeed. Picks the
    most complete protein.faa if a plan returns several assemblies."""
    tmp = Path(tempfile.mkdtemp()); zpath = tmp / "p.zip"
    plans = [("RefSeq", str(taxid), ["--reference"])]
    if species_taxid and species_taxid != taxid:
        plans.append(("RefSeq", str(species_taxid), ["--reference"]))
    plans.append(("RefSeq",  str(taxid), ["--annotated", "--assembly-version", "latest"]))
    plans.append(("GenBank", str(taxid), ["--annotated", "--assembly-version", "latest"]))
    try:
        for source, tid, extra in plans:
            if zpath.exists(): zpath.unlink()
            cmd = ["datasets", "download", "genome", "taxon", tid, "--include", "protein",
                   "--assembly-source", source, "--filename", str(zpath)] + extra
            rc, out, err = run_cmd(cmd, timeout=1200)
            if rc != 0 or not zpath.exists():
                continue
            exdir = tmp / "x"; shutil.rmtree(exdir, ignore_errors=True)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(exdir)
            faas = glob.glob(str(exdir / "**" / "protein.faa"), recursive=True)
            if not faas:
                continue
            best = max(faas, key=lambda f: os.path.getsize(f))   # most complete assembly
            dest = proteome_path(taxid)
            shutil.copy(best, dest)
            if dest.stat().st_size > 0:
                event("proteome_source", taxid=taxid, source=source, via=tid)
                return True
        return False
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

def ensure_proteome(t):
    """Guarantee proteomes/<taxid>.faa exists. Returns 'override'|'cached'|'fetched' or raises."""
    taxid = t["taxid"]; p = proteome_path(taxid)
    if t.get("proteome_file"):
        src = _resolve_input(t["proteome_file"])
        if not src.exists():
            raise FileNotFoundError(f"proteome_file not found: {src}")
        if p.resolve() != src.resolve():
            shutil.copy(src, p)
        return "override"
    if p.exists() and p.stat().st_size > 0 and not FORCE_REFETCH:
        return "cached"
    ok = retry_call(lambda: _download_proteome(taxid, t.get("species_taxid")), what=f"fetch {taxid}")
    if not ok:
        raise RuntimeError("no annotated proteome in RefSeq or GenBank "
                           "(supply a proteome_file, or substitute a same-tier relative)")
    return "fetched"

# ---- local BLAST databases ----
def db_prefix(taxid):
    return DB_DIR / str(taxid)

def db_ready(taxid):
    return bool(glob.glob(str(db_prefix(taxid)) + ".p*"))

def build_db(taxid):
    if db_ready(taxid) and not FORCE_REBUILD_DB:
        return "cached"
    for f in glob.glob(str(db_prefix(taxid)) + ".p*"):
        os.remove(f)
    rc, out, err = run_cmd(["makeblastdb", "-in", str(proteome_path(taxid)),
                            "-dbtype", "prot", "-out", str(db_prefix(taxid))])
    if rc != 0:
        raise RuntimeError(f"makeblastdb failed for {taxid}: {err[:300]}")
    return "built"

# ---- search ----
FIELDS6 = "qseqid sseqid pident length qstart qend sstart send evalue bitscore qlen slen stitle"

def run_blastp(query_fasta, taxid):
    rc, out, err = run_cmd(["blastp", "-query", str(query_fasta), "-db", str(db_prefix(taxid)),
                            "-outfmt", "6 " + FIELDS6, "-evalue", str(EVALUE),
                            "-max_target_seqs", "50", "-num_threads", "2"])
    if rc != 0:
        raise RuntimeError(f"blastp failed for {taxid}: {err[:300]}")
    return out

def select_top_hits(blast_out):
    """Best HSP per subject; filter by e-value + query coverage; keep the top TOP_N by bitscore."""
    best = {}
    for line in blast_out.splitlines():
        f = line.split("\t")
        if len(f) < 13:
            continue
        sseqid = f[1]; evalue = float(f[8]); bits = float(f[9])
        qs, qe, qlen = int(f[4]), int(f[5]), int(f[10])
        cov = 100.0 * (abs(qe - qs) + 1) / max(1, qlen)
        rec = {"sseqid": sseqid, "pident": float(f[2]), "align_length": int(f[3]),
               "evalue": evalue, "bitscore": bits, "query_coverage_pct": cov,
               "slen": int(f[11]), "stitle": f[12]}
        if sseqid not in best or bits > best[sseqid]["bitscore"]:
            best[sseqid] = rec
    kept = [r for r in best.values() if r["evalue"] <= EVALUE and r["query_coverage_pct"] >= MIN_QUERY_COVERAGE]
    kept.sort(key=lambda r: (-r["bitscore"], r["evalue"]))
    return kept[:TOP_N]

def attach_sequences(taxid, hits):
    """Pull full subject sequences from the local proteome (no network)."""
    if not hits:
        return hits
    idx = SeqIO.index(str(proteome_path(taxid)), "fasta")
    try:
        for h in hits:
            rec = idx.get(h["sseqid"])
            h["sequence"] = str(rec.seq) if rec is not None else ""
    finally:
        idx.close()
    return hits

def checkpoint_path(taxid):
    return STATE_DIR / f"{taxid}.result.json"

def load_checkpoints():
    done = {}
    for t in targets:
        p = checkpoint_path(t["taxid"])
        if p.exists():
            done[t["taxid"]] = json.loads(p.read_text(encoding="utf-8"))
    return done

def export_results(completed):
    import csv, io
    fields = ["query","taxid","label","tier","rank","sseqid","pident","evalue","bitscore",
              "query_coverage_pct","align_length","slen","stitle","fasta_id"]
    buf = io.StringIO(newline=""); w = csv.DictWriter(buf, fieldnames=fields); w.writeheader()
    fasta = []
    for t in targets:
        item = completed.get(t["taxid"])
        if not item:
            continue
        for rank, h in enumerate(item.get("hits", []), 1):
            fid = f"{h['sseqid']}__txid{t['taxid']}__{QUERY_NAME}__h{rank}"
            w.writerow({"query": QUERY_NAME, "taxid": t["taxid"], "label": t["label"],
                        "tier": t["tier"], "rank": rank, "sseqid": h["sseqid"],
                        "pident": round(h["pident"], 1), "evalue": h["evalue"],
                        "bitscore": h["bitscore"], "query_coverage_pct": round(h["query_coverage_pct"], 1),
                        "align_length": h["align_length"], "slen": h["slen"],
                        "stitle": h.get("stitle", ""), "fasta_id": fid})
            seq = h.get("sequence", "")
            if seq:
                wrapped = "\n".join(seq[i:i+80] for i in range(0, len(seq), 80))
                fasta.append(f">{fid} {t['label']} | {h.get('stitle','')}\n{wrapped}\n")
    atomic_text(SUMMARY_CSV, buf.getvalue())
    atomic_text(HITS_FASTA, "".join(fasta))


## 4. Build / refresh the proteome library

For every taxon: make sure `proteomes/<taxid>.faa` exists (override → cached → fetched) and
build its BLAST database. Both are cached on Drive, so this is cheap to re-run. **Run this again
whenever you add taxa to the CSV or drop in new proteome files** — existing ones are skipped.
Failures (usually "no reference proteome") are recorded so you can supply a `proteome_file` or
substitute a same-tier relative, without blocking the rest.

In [ ]:
import pandas as pd

status_rows = []
for i, t in enumerate(targets, 1):
    taxid = t["taxid"]
    row = {"taxid": taxid, "label": t["label"], "tier": t["tier"],
           "proteome": "", "database": "", "n_proteins": "", "error": ""}
    print(f"[{i}/{len(targets)}] {t['label']} (TaxID {taxid})")
    try:
        pstat = ensure_proteome(t); row["proteome"] = pstat
        dstat = build_db(taxid);    row["database"] = dstat
        # protein count (cheap: count '>' lines)
        n = sum(1 for ln in proteome_path(taxid).open() if ln.startswith(">"))
        row["n_proteins"] = n
        event("library_ready", taxid=taxid, proteome=pstat, database=dstat, n_proteins=n)
        print(f"    proteome={pstat}, db={dstat}, proteins={n}")
    except Exception as exc:
        row["error"] = f"{type(exc).__name__}: {exc}"
        event("library_error", taxid=taxid, error=row["error"])
        print(f"    ERROR: {row['error']}")
    status_rows.append(row)

pd.DataFrame(status_rows).to_csv(LIB_STATUS_CSV, index=False)
ok = sum(1 for r in status_rows if r["database"] and not r["error"])
print(f"\nLibrary: {ok}/{len(targets)} taxa ready; status written to {LIB_STATUS_CSV}")


## 5. Library status

A quick look at what is ready, cached, or failed. Fix any failed rows (supply a `proteome_file`,
enable `ALLOW_NONREF_FETCH`, or substitute a same-tier relative in the CSV) and re-run section 4
before searching.

In [ ]:
import pandas as pd
st = pd.read_csv(LIB_STATUS_CSV)
ready = st[(st["database"].notna()) & (st["error"].isna() | (st["error"] == ""))]
failed = st[st["error"].astype(str).str.len() > 0]
print(f"Ready: {len(ready)} / {len(st)}")
print("Proteome source:", st["proteome"].value_counts().to_dict())
if len(failed):
    print(f"\nFailed ({len(failed)}) - supply proteome_file or a same-tier substitute:")
    for _, r in failed.iterrows():
        print(f"  - {r['label']} (TaxID {r['taxid']}): {r['error']}")
else:
    print("\nNo failures.")


## 6. Search one protein query against the library

Runs `QUERY_FASTA` (a single protein, e.g. a Nuo/Ndh subunit or a cytochrome) against every
ready taxon and keeps the **top `TOP_N` hits within threshold** — deliberately not just the
single best, so paralogs (e.g. NdhD/NdhF families, the two NuoM copies of 2M complexes) and the
mutual homology of related subunits are preserved for you to sort out in the tree. Outputs
`<QUERY_NAME>_summary.csv` and `<QUERY_NAME>_hits.faa`. Checkpointed per taxon; if you change the
query or thresholds, the run resets automatically. **To search another protein, change
`QUERY_FASTA` + `QUERY_NAME` in section 1, re-run sections 2 and 6 — the library is reused.**

In [ ]:
# load the single-protein query
qpath = _resolve_input(QUERY_FASTA)
if not qpath.exists():
    raise FileNotFoundError(f"Query FASTA not found at {qpath}. Put it in {INPUT_DIR}.")
qrecs = list(SeqIO.parse(qpath, "fasta"))
if not qrecs:
    raise ValueError(f"No sequence in {qpath}")
if len(qrecs) > 1:
    print(f"NOTE: {len(qrecs)} sequences in query; using the first ({qrecs[0].id}).")
query_seq = str(qrecs[0].seq)

# signature guard: changing query/thresholds invalidates old checkpoints for this query
sig = hashlib.sha256(("|".join([QUERY_NAME, query_seq, str(EVALUE),
                                str(MIN_QUERY_COVERAGE), str(TOP_N)])).encode()).hexdigest()
manifest_path = RUNS_DIR / "manifest.json"
old = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
if old.get("signature") != sig:
    for f in STATE_DIR.glob("*.result.json"):
        f.unlink()
    save_json(manifest_path, {"signature": sig, "query": QUERY_NAME, "created": utc_now(),
                              "evalue": EVALUE, "min_coverage": MIN_QUERY_COVERAGE, "top_n": TOP_N})
    print("New/changed query or thresholds - previous checkpoints for this query cleared.")

st = pd.read_csv(LIB_STATUS_CSV)
ready_ids = set(int(x) for x in st[(st["error"].isna()) | (st["error"] == "")]["taxid"])
ready = [t for t in targets if t["taxid"] in ready_ids and db_ready(t["taxid"])]

completed = load_checkpoints()
export_results(completed)
pending = [t for t in ready if t["taxid"] not in completed]
print(f"{len(completed)}/{len(ready)} taxa already searched for {QUERY_NAME}; {len(pending)} to go.")

for i, t in enumerate(pending, 1):
    taxid = t["taxid"]
    print(f"[{i}/{len(pending)}] {t['label']} (TaxID {taxid})")
    try:
        out = run_blastp(qpath, taxid)
        hits = attach_sequences(taxid, select_top_hits(out))
        item = {"taxid": taxid, "label": t["label"], "status": "hits" if hits else "no_hits",
                "n_hits": len(hits), "hits": hits, "completed": utc_now()}
        save_json(checkpoint_path(taxid), item)
        completed[taxid] = item
        export_results(completed)
        event("searched", taxid=taxid, query=QUERY_NAME, n_hits=len(hits))
        print(f"    {len(hits)} hit(s)" + (": " + ", ".join(h["sseqid"] for h in hits) if hits else ""))
    except Exception as exc:
        event("search_error", taxid=taxid, query=QUERY_NAME, error=f"{type(exc).__name__}: {exc}")
        print(f"    ERROR: {type(exc).__name__}: {exc}")

completed = load_checkpoints()
export_results(completed)
total_hits = sum(len(v.get("hits", [])) for v in completed.values())
print(f"\n{QUERY_NAME}: searched {len(completed)}/{len(ready)} taxa, {total_hits} hits total.")
print(f"  table:  {SUMMARY_CSV}")
print(f"  FASTA:  {HITS_FASTA}")


## 7. Swapping, growing, and re-running

- **Another protein:** change `QUERY_FASTA` + `QUERY_NAME` (section 1), re-run sections 2 and 6.
  New `QUERY_NAME` -> fresh `Data/<COLLECTION>/<QUERY_NAME>/` and `Outputs/<COLLECTION>/<QUERY_NAME>/`
  subfolders. The proteome library is shared and is *not* rebuilt.
- **Add taxa:** add rows to the shared taxon table in `Databases/proteome_library/` (or drop
  `<taxid>.faa` into `Databases/proteome_library/proteomes/`), re-run section 4, then re-run
  section 6 for each protein (only new taxa are searched).
- **Swap one proteome** (a curated or better assembly): drop your `<taxid>.faa` into
  `Databases/proteome_library/proteomes/` (or set a `proteome_file` column in the table), delete
  that taxon's `Databases/proteome_library/blastdbs/<taxid>.p*` (or set `FORCE_REBUILD_DB = True`),
  re-run section 4.
- **Re-search one taxon:** delete `Outputs/<COLLECTION>/<QUERY_NAME>/state/<taxid>.result.json` and
  re-run section 6.
- **Rebuild everything:** set `FORCE_REFETCH` / `FORCE_REBUILD_DB = True` for one run, then back to False.

Per-protein outputs live in `Outputs/<COLLECTION>/<QUERY_NAME>/`: `*_summary.csv`, `*_hits.faa`,
`events.jsonl`. The shared library lives in `Databases/proteome_library/`. Version the code by
committing this notebook to your repo yourself.